In [1]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine, sigmoid
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix

In [2]:
config_file = '../../Dataset/JSON/full_model_configs.json'

model_configs = []

try:
    with open(config_file, 'r') as f:
        configs = json.load(f)
        for config in configs:
            if config["Activation"] == "sigmoid":
                config["Activation"] = sigmoid
        model_configs.extend(configs)
except FileNotFoundError:
    print(f"Error: {config_file} not found. Run ELM_Evaluation and Grid_Evaluation first.")

print(f"Loaded {len(model_configs)} configurations ready for testing.")

Loaded 5 configurations ready for testing.


In [3]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
targetCol = ['Gallstone Status']
# filePath = '../../Dataset/DTC_R.csv'
# targetCol = ['Recurred_Yes']
df = pd.read_csv(filePath)


X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)


main_scaler = MinMaxScaler()
x_train_scaled = pd.DataFrame(main_scaler.fit_transform(x_train), columns=X.columns)
x_test_scaled  = pd.DataFrame(main_scaler.transform(x_test), columns=X.columns)

In [5]:
testing_results = []
for config in model_configs:
    print(f"Executing: Nodes : {config['Hidden_Nodes']} , lambda value : {config['Lambda_Value']} , activation function : {config['Activation'].__name__}")

    elm = ExtremeLearningMachine(
        features_size=features_size,
        hidden_size=config["Hidden_Nodes"],
        activation_function=config["Activation"],
        regularization_lambda=config["Lambda_Value"]
    )
    elm.initialize_random_weights(random_seed=42)


    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())

    metrics = evaluation.get_all_metrics()
    test_result = {
        "Model_Type"   : config.get('Model_Types', 'Unknown'), # Extract the strategy name
        "Hidden_Nodes" : config['Hidden_Nodes'],
        "Lambda_Value" : config['Lambda_Value'],
        "Activation"   : config['Activation'].__name__
    }
    test_result.update(metrics)
    testing_results.append(test_result)
    print("\n")

Executing: Nodes : 56 , lambda value : 0.0 , activation function : sigmoid
{'Counts': {'TP': np.int64(21), 'TN': np.int64(25), 'FP': np.int64(7), 'FN': np.int64(11)}, 'Metrics': {'Accuracy': 0.7188, 'Precision': 0.75, 'Recall': 0.6562, 'NPV': 0.6944, 'Specificity': 0.7812, 'F1-Score': 0.7, 'F2-Score': 0.6731, 'Balanced Accuracy': 0.7188, 'MCC': 0.441}}


Executing: Nodes : 56 , lambda value : 1.25 , activation function : sigmoid
{'Counts': {'TP': np.int64(20), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(12)}, 'Metrics': {'Accuracy': 0.6875, 'Precision': 0.7143, 'Recall': 0.625, 'NPV': 0.6667, 'Specificity': 0.75, 'F1-Score': 0.6667, 'F2-Score': 0.641, 'Balanced Accuracy': 0.6875, 'MCC': 0.378}}


Executing: Nodes : 58 , lambda value : 1.0 , activation function : sigmoid
{'Counts': {'TP': np.int64(27), 'TN': np.int64(23), 'FP': np.int64(9), 'FN': np.int64(5)}, 'Metrics': {'Accuracy': 0.7812, 'Precision': 0.75, 'Recall': 0.8438, 'NPV': 0.8214, 'Specificity': 0.7188, 'F1-Score':

In [6]:
final_model_result = pd.DataFrame(testing_results)
final_model_result

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy,Precision,Recall,NPV,Specificity,F1-Score,F2-Score,Balanced Accuracy,MCC
0,Best_Hidden_Nodes,56,0.00,sigmoid,0.71875,0.750000,0.65625,0.694444,0.78125,0.700000,0.673077,0.71875,0.440959
1,Best_Lambda,56,1.25,sigmoid,0.68750,0.714286,0.62500,0.666667,0.75000,0.666667,0.641026,0.68750,0.377964
2,Best_Size_and_Lambda,58,1.00,sigmoid,0.78125,0.750000,0.84375,0.821429,0.71875,0.794118,0.823171,0.78125,0.566947
3,Grid_Combination,58,1.00,sigmoid,0.78125,0.750000,0.84375,0.821429,0.71875,0.794118,0.823171,0.78125,0.566947
4,Grid_Optimization,50,0.25,sigmoid,0.78125,0.800000,0.75000,0.764706,0.81250,0.774194,0.759494,0.78125,0.563602
